# Feature Engineering สำหรับ Dataset กลาง

Notebook นี้สร้าง model features จากผลการรวม/split ใน `common/output/` และเก็บ labels กับ metadata แยกไว้ ไม่ import ฟังก์ชันจากไฟล์ Python ภายนอก

โปรไฟล์ joint_v1 ใช้ 19 model features: ตัวเลข 4, protocol 4, service 7 และ connection state 4; `record_id` เก็บไว้เชื่อมตารางแต่ไม่ใช่ feature

หลักสำคัญ: เรียนรู้ scaler/encoder จาก `train` เท่านั้น แล้วใช้ตัวแปลงเดิมกับ `train_full`, `validation` และ `test`; การ fit ไม่ใช้ `y`

## 1. เตรียม path และการตรวจตาราง

หา project root อัตโนมัติ แล้วกำหนดตำแหน่ง input/output รวมถึงตรวจว่า `record_id` ของสามตารางตรงกัน.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_project_root(start=None):
    """หาโฟลเดอร์โปรเจกต์จากตำแหน่งที่เปิด notebook โดยไม่ล็อก path เครื่องใดเครื่องหนึ่ง."""
    origin = Path(start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        has_pipeline = (candidate / "data-pipeline").is_dir()
        has_project_marker = (candidate / ".git").exists() or (candidate / "README.md").is_file()
        if has_pipeline and has_project_marker:
            return candidate
    raise FileNotFoundError(f"หาโฟลเดอร์โปรเจกต์จากตำแหน่งนี้ไม่พบ: {origin}")


PROJECT_ROOT = find_project_root()
COMMON_ROOT = PROJECT_ROOT / "data-pipeline" / "common"
BUILD_ROOT = COMMON_ROOT / "output"
OUTPUT_ROOT = BUILD_ROOT / "feature_engineering"
SPLITS = ("train", "train_full", "validation", "test")
KEY = "record_id"
FORBIDDEN_FEATURE_COLUMNS = {
    "label", "label_raw", "attack_category", "label_binary", "is_attack",
    "source_dataset", "orig_row_id", "capture_time", "split_group",
    "sample_weight", "dedup_flag", "data_split", "difficulty_level",
}


def validate_aligned_tables(features, labels, metadata):
    """ตรวจการเชื่อม features/labels/metadata และป้องกันข้อมูลเป้าหมายรั่วเข้า X."""
    tables = {"features": features, "labels": labels, "metadata": metadata}
    for table_name, frame in tables.items():
        if KEY not in frame.columns:
            raise ValueError(f"ตาราง {table_name} ไม่มีคอลัมน์ {KEY!r}")
        if frame[KEY].isna().any() or frame[KEY].astype(str).eq("").any():
            raise ValueError(f"{table_name}.{KEY} มีค่าว่าง")
        if frame[KEY].duplicated().any():
            examples = frame.loc[frame[KEY].duplicated(), KEY].head(3).tolist()
            raise ValueError(f"{table_name}.{KEY} ซ้ำ ตัวอย่าง: {examples}")
    reference_ids = features[KEY].astype(str).reset_index(drop=True)
    for table_name, frame in (("labels", labels), ("metadata", metadata)):
        if not reference_ids.equals(frame[KEY].astype(str).reset_index(drop=True)):
            raise ValueError(f"ลำดับ record_id ใน features และ {table_name} ไม่ตรงกัน")
    leaked_columns = FORBIDDEN_FEATURE_COLUMNS.intersection(features.columns)
    if leaked_columns:
        raise ValueError(f"พบ label/metadata ปะปนใน features: {sorted(leaked_columns)}")
    if set(labels.columns) == {KEY}:
        raise ValueError("ตาราง labels ไม่มีคอลัมน์ label")
    return {name: len(frame) for name, frame in tables.items()}


print(f"โฟลเดอร์โปรเจกต์: {PROJECT_ROOT}")
print(f"ข้อมูลกลางที่อ่าน: {BUILD_ROOT}")
print(f"ผล Feature Engineering: {OUTPUT_ROOT}")


โฟลเดอร์โปรเจกต์: C:\Users\Windows 10\Desktop\CSI_ML_Final
ข้อมูลกลางที่อ่าน: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\common\output
ผล Feature Engineering: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\common\output\feature_engineering


## 2. กำหนดหมวดหมู่และคำนวณ feature ตัวเลข

ใช้ `log1p` ลดช่วงของ duration และ bytes ส่วนอัตราส่วนแบบมีเครื่องหมายคำนวณจาก bytes สองทิศทางอย่างสมมาตร: ค่าบวกหมายถึงขาไปมากกว่า ค่าลบหมายถึงขากลับมากกว่า และศูนย์หมายถึงใกล้เคียงกัน

In [2]:
NUMERIC_INPUTS = ["duration", "bytes_forward", "bytes_backward"]
CATEGORICAL_INPUTS = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}
CATEGORICAL_OUTPUT_PREFIXES = {"protocol": "protocol", "service_group": "service", "conn_state_group": "conn_state"}
CATEGORICAL_OUTPUT_CATEGORY_ALIASES = {"conn_state_group": {"no-reply": "no_reply"}}

def _model_category_name(input_column: str, encoded_name: str) -> str:
    category = encoded_name.removeprefix(input_column + "_")
    category = CATEGORICAL_OUTPUT_CATEGORY_ALIASES.get(input_column, {}).get(category, category)
    return f"{CATEGORICAL_OUTPUT_PREFIXES[input_column]}_{category}"


def _derive(frame: pd.DataFrame) -> pd.DataFrame:
    """Create numeric traffic features without changing the source table."""
    duration = frame["duration"].astype(float)
    forward = frame["bytes_forward"].astype(float)
    backward = frame["bytes_backward"].astype(float)
    return pd.DataFrame({
        "log_duration": np.log1p(duration),
        "log_bytes_forward": np.log1p(forward),
        "log_bytes_backward": np.log1p(backward),
        "signed_log_bytes_ratio": np.log1p(forward) - np.log1p(backward),
    }, index=frame.index)


def _normalize_categories(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize categories; unknown values become 'other', missing values are errors."""
    normalized = {}
    for column, categories in CATEGORICAL_INPUTS.items():
        values = frame[column].astype("string").str.strip().str.lower()
        missing = values.isna() | values.eq("")
        if missing.any():
            raise ValueError(f"{column} has {int(missing.sum()):,} missing values")
        normalized[column] = values.where(values.isin(categories), "other")
    return pd.DataFrame(normalized, index=frame.index)

## 3. ตรวจความพร้อม เลือก feature และอ่าน split

อ่านรายงานความเข้ากันได้จากขั้น Build ก่อนเลือก core features หากมี dataset เดียวพร้อม จะระบุขอบเขตเป็น `source_local_diagnostic` ไม่อ้างว่าเป็น feature ที่ยืนยันร่วมกันข้ามสอง dataset.

In [3]:
def run_feature_engineering() -> dict:
    """สร้าง model input โดยเรียนรู้ scaler/encoder จาก Train เท่านั้น."""
    compatibility_path = BUILD_ROOT / "reports" / "feature_compatibility.json"
    if not compatibility_path.is_file():
        raise FileNotFoundError("Run common/build_dataset.ipynb before central feature engineering.")
    compatibility_checks = json.loads(compatibility_path.read_text(encoding="utf-8"))
    usable_core_features = [
        row["feature"]
        for row in compatibility_checks
        if row.get("usable_in_current_build", row["eligible_for_joint_profile"])
    ]
    required_numeric = {"duration", "bytes_forward", "bytes_backward"}
    if not required_numeric.issubset(usable_core_features):
        missing = sorted(required_numeric.difference(usable_core_features))
        raise ValueError(f"Shared profile is missing required numeric inputs: {missing}")

    readiness_path = BUILD_ROOT / "reports" / "dataset_readiness.csv"
    if readiness_path.is_file():
        readiness = pd.read_csv(readiness_path)
        joint_profile_ready = bool(len(readiness)) and bool(readiness["ready"].astype(bool).all())
    else:
        joint_profile_ready = all(
            row.get("eligible_for_joint_profile", False) for row in compatibility_checks
        )
    profile_scope = "joint" if joint_profile_ready else "source_local_diagnostic"

    def read_split(name: str):
        folder = BUILD_ROOT / "splits" / name
        features = pd.read_csv(folder / "features.csv")
        labels = pd.read_csv(folder / "labels.csv", dtype={"label_raw": "string"})
        metadata = pd.read_csv(folder / "metadata.csv")
        validate_aligned_tables(features, labels, metadata)
        return features, labels, metadata

    splits = {name: read_split(name) for name in SPLITS}
    # ตรวจ input ของทุก split; ไม่เติมศูนย์หรือทิ้งแถว
    required_inputs = set(NUMERIC_INPUTS) | set(CATEGORICAL_INPUTS)
    for name, (features, _, _) in splits.items():
        missing_columns = sorted(required_inputs.difference(features.columns))
        if missing_columns:
            raise ValueError(f"{name} is missing required shared inputs: {missing_columns}")
        missing_counts = features[list(required_inputs)].isna().sum()
        if missing_counts.any():
            raise ValueError(
                f"{name} has missing required inputs: {missing_counts[missing_counts > 0].to_dict()}"
            )

    # Fit preprocessing จาก Train ที่สุ่มให้สมดุลแหล่งข้อมูลเท่านั้น
    numeric_train = _derive(splits["train"][0])
    categorical_train = _normalize_categories(splits["train"][0])
    if numeric_train.isna().any().any() or not np.isfinite(numeric_train.to_numpy(dtype=float)).all():
        raise ValueError("Derived training features contain missing or infinite values.")

    # StandardScaler และ OneHotEncoder เรียนรู้ค่าจาก Train แล้ว reuse กับ split อื่น
    scaler = StandardScaler()
    scaler.fit(numeric_train)  # fit บน Train; ยังไม่ต้องสร้าง transformed array ที่ไม่ได้ใช้
    encoders = {}
    categorical_feature_groups = {}
    for column, categories in CATEGORICAL_INPUTS.items():
        encoder = OneHotEncoder(
            categories=[categories],
            handle_unknown="ignore",
            sparse_output=False,
        )
        encoder.fit(categorical_train[[column]])
        encoders[column] = encoder
        encoded_names = encoder.get_feature_names_out([column]).tolist()
        categorical_feature_groups[column] = [
            _model_category_name(column, name) for name in encoded_names
        ]
    numeric_names = numeric_train.columns.tolist()
    categorical_names = [name for names in categorical_feature_groups.values() for name in names]
    output_names = numeric_names + categorical_names
    if len(output_names) != 19:
        raise ValueError(f"Expected 19 model features, got {len(output_names)}")

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    import joblib
    preprocessor_path = OUTPUT_ROOT / "preprocessor.joblib"
    joblib.dump({"scaler": scaler, "encoders": encoders,
                 "usable_core_features": usable_core_features,
                 "categorical_inputs": CATEGORICAL_INPUTS,
                 "categorical_output_prefixes": CATEGORICAL_OUTPUT_PREFIXES,
                 "categorical_output_category_aliases": CATEGORICAL_OUTPUT_CATEGORY_ALIASES,
                 "categorical_feature_groups": categorical_feature_groups,
                 "numeric_feature_names": numeric_names, "model_feature_names": output_names,
                 "fit_split": "train"}, preprocessor_path)

    # แปลงและบันทึกแต่ละ split; label/metadata ไม่ถูกรวมเข้า model features
    reports = []
    for name, (features, labels, metadata) in splits.items():
        derived = _derive(features)
        if derived.isna().any().any() or not np.isfinite(derived.to_numpy(dtype=float)).all():
            raise ValueError(f"{name} has missing/infinite engineered values; no records were silently dropped.")
        numeric_scaled = scaler.transform(derived[numeric_names])
        categorical_values = _normalize_categories(features)
        categorical_blocks = [
            encoders[column].transform(categorical_values[[column]])
            for column in CATEGORICAL_INPUTS
        ]
        matrix = np.concatenate([numeric_scaled, *categorical_blocks], axis=1)
        model_features = pd.DataFrame(matrix, columns=output_names)
        for group_name, group_columns in categorical_feature_groups.items():
            group_total = model_features[group_columns].sum(axis=1)
            if not group_total.eq(1).all():
                raise ValueError(f"{name}: one-hot group {group_name} must have exactly one active value per row")
        model_features.insert(0, "record_id", features["record_id"].astype(str).to_numpy())
        target_output = labels.copy()
        metadata_output = metadata.copy()
        validate_aligned_tables(model_features, target_output, metadata_output)
        folder = OUTPUT_ROOT / name
        folder.mkdir(parents=True, exist_ok=True)
        model_features.to_csv(folder / "features.csv", index=False, encoding="utf-8")
        target_output.to_csv(folder / "labels.csv", index=False, encoding="utf-8")
        metadata_output.to_csv(folder / "metadata.csv", index=False, encoding="utf-8")
        reports.append({"split": name, "profile_scope": profile_scope,
                        "rows": len(model_features), "model_features": len(output_names),
                        "source_datasets": ",".join(
                            sorted(metadata["source_dataset"].unique())
                        )})

    schema = {"profile_name": "joint_v1", "expected_model_feature_count": 19,
              "feature_names": output_names, "numeric_features": numeric_names,
              "categorical_features": categorical_names,
              "categorical_inputs": list(CATEGORICAL_INPUTS),
              "categorical_output_prefixes": CATEGORICAL_OUTPUT_PREFIXES,
              "categorical_output_category_aliases": CATEGORICAL_OUTPUT_CATEGORY_ALIASES,
              "categorical_feature_groups": categorical_feature_groups,
              "source_features_used": NUMERIC_INPUTS + list(CATEGORICAL_INPUTS),
              "feature_formulas": {
                  "log_duration": "log1p(duration)",
                  "log_bytes_forward": "log1p(bytes_forward)",
                  "log_bytes_backward": "log1p(bytes_backward)",
                  "signed_log_bytes_ratio": "log1p(bytes_forward) - log1p(bytes_backward)"
              },
              "core_features_used": usable_core_features,
              "profile_scope": profile_scope, "joint_profile_ready": joint_profile_ready,
              "source_datasets": sorted(splits["train"][2]["source_dataset"].astype(str).unique()),
              "fit_on": "train", "fit_records": len(splits["train"][0]),
              "source_dataset_is_not_a_model_feature": True,
              "metadata_and_targets_are_saved_separately": True,
              "note": "record_id is included in the feature table only for joins; drop it before passing X to a model."}
    (OUTPUT_ROOT / "feature_schema.json").write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(reports).to_csv(OUTPUT_ROOT / "feature_engineering_report.csv", index=False)
    return {"output_directory": str(OUTPUT_ROOT), "preprocessor": str(preprocessor_path),
            "core_features_used": usable_core_features,
            "engineered_features": output_names,
            "profile_scope": profile_scope, "joint_profile_ready": joint_profile_ready,
            "fit_rows": len(splits["train"][0]), "splits": reports}

## 4. สร้างผลลัพธ์

รันหลังจากทำ Build Dataset กลางแล้ว ระบบจะ fit preprocessing บน Train และส่งผลลัพธ์แยกตาม split ไปที่ `common/output/feature_engineering/` พร้อม schema และรายงานจำนวนแถว/feature.

In [4]:
feature_engineering_summary = run_feature_engineering()
display(pd.DataFrame(feature_engineering_summary["splits"]))
print("ขอบเขตโปรไฟล์:", feature_engineering_summary["profile_scope"])
print("Core features:", feature_engineering_summary["core_features_used"])
print("ไฟล์ผลลัพธ์:", feature_engineering_summary["output_directory"])

,split,profile_scope,rows,model_features,source_datasets
0,train,joint,201556,19,"NSL-KDD,UNSW-NB15"
1,train_full,joint,241051,19,"NSL-KDD,UNSW-NB15"
2,validation,joint,60263,19,"NSL-KDD,UNSW-NB15"
3,test,joint,104876,19,"NSL-KDD,UNSW-NB15"


ขอบเขตโปรไฟล์: joint
Core features: ['duration', 'protocol', 'bytes_forward', 'bytes_backward']
ไฟล์ผลลัพธ์: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\common\output\feature_engineering


## 5. แสดงสูตร ตัวอย่างก่อน-หลัง และผลของแต่ละ split

ตารางสูตรช่วยตอบว่า feature ใหม่มาจากคอลัมน์ใด ส่วนตัวอย่างแสดงค่าต้นฉบับ → ค่าที่คำนวณ → ค่าหลัง StandardScaler จริง สรุป split ตรวจว่าไม่มีแถวถูกทิ้งและไม่มี label/metadata ปนใน X ค่าเฉลี่ย/ส่วนเบี่ยงเบนที่ใกล้ 0/1 ควรเห็นเด่นใน Train เพราะ scaler fit จาก Train เท่านั้น; Validation/Test อาจไม่เป็น 0/1

In [5]:
feature_schema = json.loads((OUTPUT_ROOT / 'feature_schema.json').read_text(encoding='utf-8'))
used_core = feature_schema['core_features_used']
used_inputs = feature_schema['source_features_used']
numeric_names = feature_schema['numeric_features']
recipe_rows = []
recipe_by_name = {
    'log_duration': ('duration', 'log(1 + duration)', 'บีบช่วง duration ที่มีหางยาว'),
    'log_bytes_forward': ('bytes_forward', 'log(1 + bytes_forward)', 'บีบช่วง bytes ทิศทางต้นทาง'),
    'log_bytes_backward': ('bytes_backward', 'log(1 + bytes_backward)', 'บีบช่วง bytes ทิศทางกลับ'),
    'signed_log_bytes_ratio': ('bytes_forward, bytes_backward', 'log1p(forward) - log1p(backward)', 'ค่าบวกคือขาไปมากกว่า ค่าลบคือขากลับมากกว่า'),
}
for output_name in numeric_names:
    source, formula, purpose = recipe_by_name.get(output_name, ('core feature', 'คำนวณตามสูตรใน cell ก่อนหน้า', 'feature เชิงตัวเลขที่ใช้ร่วมได้'))
    recipe_rows.append({'model_feature': output_name, 'source_feature': source,
                        'formula_before_scaling': formula, 'เหตุผล': purpose,
                        'หลังคำนวณ': 'StandardScaler ที่ fit บน Train'})
for source_column, output_columns in feature_schema['categorical_feature_groups'].items():
    for output_name in output_columns:
        category_name = output_name.removeprefix(source_column + '_')
        recipe_rows.append({'model_feature': output_name, 'source_feature': source_column,
                            'formula_before_scaling': f'One-Hot: {source_column} = {category_name}',
                            'เหตุผล': 'แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม',
                            'หลังคำนวณ': 'ไม่ผ่าน StandardScaler'})
display(pd.DataFrame(recipe_rows))

split_audit_rows, statistic_rows = [], []
for split_name in SPLITS:
    input_dir = BUILD_ROOT / 'splits' / split_name
    output_dir = OUTPUT_ROOT / split_name
    source_features = pd.read_csv(input_dir / 'features.csv')
    source_labels = pd.read_csv(input_dir / 'labels.csv', dtype={'label_raw': 'string'}, low_memory=False)
    source_metadata = pd.read_csv(input_dir / 'metadata.csv')
    model_features = pd.read_csv(output_dir / 'features.csv')
    output_labels = pd.read_csv(output_dir / 'labels.csv', dtype={'label_raw': 'string'}, low_memory=False)
    output_metadata = pd.read_csv(output_dir / 'metadata.csv')
    if not source_features[KEY].astype(str).reset_index(drop=True).equals(model_features[KEY].astype(str).reset_index(drop=True)):
        raise ValueError(f'{split_name}: record_id ก่อน/หลัง feature engineering ไม่ตรงกัน')
    split_audit_rows.append({
        'split': split_name, 'input_rows': len(source_features), 'output_rows': len(model_features),
        'rows_removed': len(source_features) - len(model_features),
        'input_core_features': len(used_core),
        'input_shared_features': len(used_inputs),
        'output_model_features_excluding_record_id': model_features.shape[1] - 1,
        'labels_rows_separate': len(output_labels), 'metadata_rows_separate': len(output_metadata),
        'missing_model_values': int(model_features.drop(columns=[KEY]).isna().sum().sum()),
        'record_ids_unique': bool(model_features[KEY].is_unique),
        'source_feature_cols_with_label_metadata': len(FORBIDDEN_FEATURE_COLUMNS.intersection(source_features.columns)),
        'input_metadata_rows': len(source_metadata),
    })
    for column in model_features.columns:
        if column == KEY:
            continue
        statistic_rows.append({
            'split': split_name, 'feature': column,
            'mean_after_transform': model_features[column].mean(),
            'std_after_transform': model_features[column].std(),
            'min': model_features[column].min(), 'max': model_features[column].max(),
        })
print('ตรวจจำนวนข้อมูลและการแยก X / y / metadata:')
display(pd.DataFrame(split_audit_rows))
print('สถิติของค่าที่ส่งเข้าโมเดลหลังแปลง (สถิติ split ใหญ่คำนวณจากทุกแถว):')
display(pd.DataFrame(statistic_rows))

train_input = pd.read_csv(BUILD_ROOT / 'splits' / 'train' / 'features.csv', nrows=5)
train_output = pd.read_csv(OUTPUT_ROOT / 'train' / 'features.csv', nrows=5)
engineered_preview = _derive(train_input)
example = pd.DataFrame({'record_id': train_input[KEY].astype(str)})
for column in used_inputs:
    if column in train_input:
        example[f'ต้นทาง: {column}'] = train_input[column]
for column in engineered_preview:
    example[f'คำนวณก่อน scale: {column}'] = engineered_preview[column]
for column in train_output.columns:
    if column != KEY:
        example[f'เข้าโมเดลหลัง scale: {column}'] = train_output[column]
print('ตัวอย่าง 5 แถว: ต้นทาง → feature ที่คำนวณ → ค่าที่เข้าโมเดล')
display(example)
print('สำคัญ: label ใช้ประเมินภายหลังเท่านั้น; ตัว fit scaler/encoder ใช้ X ของ train ไม่ใช้ y')

,model_feature,source_feature,formula_before_scaling,เหตุผล,หลังคำนวณ
0,log_duration,duration,log(1 + duration),บีบช่วง duration ที่มีหางยาว,StandardScaler ที่ fit บน Train
1,log_bytes_forward,bytes_forward,log(1 + bytes_forward),บีบช่วง bytes ทิศทางต้นทาง,StandardScaler ที่ fit บน Train
2,log_bytes_backward,bytes_backward,log(1 + bytes_backward),บีบช่วง bytes ทิศทางกลับ,StandardScaler ที่ fit บน Train
3,signed_log_bytes_ratio,"bytes_forward, bytes_backward",log1p(forward) - log1p(backward),ค่าบวกคือขาไปมากกว่า ค่าลบคือขากลับมากกว่า,StandardScaler ที่ fit บน Train
4,protocol_tcp,protocol,One-Hot: protocol = tcp,แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม,ไม่ผ่าน StandardScaler
5,protocol_udp,protocol,One-Hot: protocol = udp,แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม,ไม่ผ่าน StandardScaler
6,protocol_icmp,protocol,One-Hot: protocol = icmp,แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม,ไม่ผ่าน StandardScaler
7,protocol_other,protocol,One-Hot: protocol = other,แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม,ไม่ผ่าน StandardScaler
8,service_dns,service_group,One-Hot: service_group = service_dns,แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม,ไม่ผ่าน StandardScaler
9,service_ftp,service_group,One-Hot: service_group = service_ftp,แปลงหมวดข้อความเป็นตัวเลขโดยไม่สร้างลำดับปลอม,ไม่ผ่าน StandardScaler


ตรวจจำนวนข้อมูลและการแยก X / y / metadata:


,split,input_rows,output_rows,rows_removed,input_core_features,input_shared_features,output_model_features_excluding_record_id,labels_rows_separate,metadata_rows_separate,missing_model_values,record_ids_unique,source_feature_cols_with_label_metadata,input_metadata_rows
0,train,201556,201556,0,4,6,19,201556,201556,0,True,0,201556
1,train_full,241051,241051,0,4,6,19,241051,241051,0,True,0,241051
2,validation,60263,60263,0,4,6,19,60263,60263,0,True,0,60263
3,test,104876,104876,0,4,6,19,104876,104876,0,True,0,104876


สถิติของค่าที่ส่งเข้าโมเดลหลังแปลง (สถิติ split ใหญ่คำนวณจากทุกแถว):


,split,feature,mean_after_transform,std_after_transform,min,max
0,train,log_duration,2.143374e-17,1.000002,-0.294280,9.177361
1,train,log_bytes_forward,2.887915e-16,1.000002,-1.666798,5.703748
2,train,log_bytes_backward,1.263463e-16,1.000002,-0.914965,4.636787
3,train,signed_log_bytes_ratio,3.609894e-17,1.000002,-7.374765,6.532251
4,train,protocol_tcp,6.360218e-01,0.481144,0.000000,1.000000
...,...,...,...,...,...,...
71,test,service_other,5.748122e-01,0.494374,0.000000,1.000000
72,test,conn_state_established,5.835081e-01,0.492979,0.000000,1.000000
73,test,conn_state_no_reply,3.625043e-01,0.480726,0.000000,1.000000
74,test,conn_state_reset,5.048820e-02,0.218951,0.000000,1.000000


ตัวอย่าง 5 แถว: ต้นทาง → feature ที่คำนวณ → ค่าที่เข้าโมเดล


,record_id,ต้นทาง: duration,ต้นทาง: bytes_forward,ต้นทาง: bytes_backward,ต้นทาง: protocol,ต้นทาง: service_group,ต้นทาง: conn_state_group,คำนวณก่อน scale: log_duration,คำนวณก่อน scale: log_bytes_forward,คำนวณก่อน scale: log_bytes_backward,...,เข้าโมเดลหลัง scale: service_ftp,เข้าโมเดลหลัง scale: service_http,เข้าโมเดลหลัง scale: service_https_ssl,เข้าโมเดลหลัง scale: service_smtp,เข้าโมเดลหลัง scale: service_ssh,เข้าโมเดลหลัง scale: service_other,เข้าโมเดลหลัง scale: conn_state_established,เข้าโมเดลหลัง scale: conn_state_no_reply,เข้าโมเดลหลัง scale: conn_state_reset,เข้าโมเดลหลัง scale: conn_state_other
0,NSL-KDD:KDDTrain:000000000,0.0,491.0,0.0,tcp,ftp,established,0.0,6.198479,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,NSL-KDD:KDDTrain:000000001,0.0,146.0,0.0,udp,other,established,0.0,4.990433,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,NSL-KDD:KDDTrain:000000002,0.0,0.0,0.0,tcp,other,no-reply,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,NSL-KDD:KDDTrain:000000003,0.0,232.0,8153.0,tcp,http,established,0.0,5.451038,9.006264,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,NSL-KDD:KDDTrain:000000004,0.0,199.0,420.0,tcp,http,established,0.0,5.298317,6.042633,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


สำคัญ: label ใช้ประเมินภายหลังเท่านั้น; ตัว fit scaler/encoder ใช้ X ของ train ไม่ใช้ y
